# Experiment 10 — AdaBoost × 5 Imbalance Techniques

AdaBoost (Adaptive Boosting) works by sequentially fitting simple decision tree estimators on adjusted weights.

**AdaBoost limitations:**
- Does NOT support `scale_pos_weight` — uses `sample_weight` instead.
- Does NOT support custom objective (focal loss) — skip focal loss for AdaBoost, replace with `class_weight='balanced'` equivalent.
- Is significantly slower than tree ensemble methods on large datasets.
- We limit the estimators and depth to keep runs reasonable for CPU benchmark.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Standard AdaBoost settings (constrained for reasonable speed)
ADA_PARAMS = dict(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=100, random_state=42)
SMOTE_CAP  = 500_000
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 10 — AdaBoost × 5 Techniques")

Experiment 10 — AdaBoost × 5 Techniques


In [2]:
def run_ada_technique(X_train, y_train, X_test, y_test, technique, v):
    t0 = time.time()

    if technique == 'baseline':
        model = AdaBoostClassifier(**ADA_PARAMS)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'class_weights' or technique == 'focal_loss':
        # If focal_loss, replace with class_weight='balanced' equivalent via sample_weight
        if technique == 'focal_loss':
            print("    [NOTE] AdaBoost does not support focal loss — replacing with class_weight='balanced' equivalent via sample_weight.")
        n_bg  = (y_train==0).sum(); n_sig = (y_train==1).sum()
        total = n_bg + n_sig
        w_bg  = total / (2.0 * n_bg)
        w_sig = total / (2.0 * n_sig)
        sample_weights = np.where(y_train == 1, w_sig, w_bg)

        model = AdaBoostClassifier(**ADA_PARAMS)
        model.fit(X_train, y_train, sample_weight=sample_weights)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'smote':
        if v == 'A' and len(X_train) > SMOTE_CAP:
            idx = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP, replace=False)
            X_s, y_s = X_train[idx], y_train[idx]
        else:
            X_s, y_s = X_train, y_train
        X_res, y_res = SMOTE(random_state=42).fit_resample(X_s, y_s)
        smote_t = time.time() - t0
        model   = AdaBoostClassifier(**ADA_PARAMS)
        model.fit(X_res, y_res)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0+smote_t), probs

    elif technique == 'threshold':
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model = AdaBoostClassifier(**ADA_PARAMS)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s    = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                  for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        m = compute_all_metrics(y_test, probs, preds, time.time()-t0)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

In [3]:
techniques = ['baseline','class_weights','smote','focal_loss','threshold']
tech_names = {'baseline':'Baseline','class_weights':'Class Weights',
              'smote':'SMOTE','focal_loss':'Focal Loss','threshold':'Threshold Opt.'}
all_results = []

for v in ['A','B','C']:
    print(f"\n[Exp10-ADA] Version {v}")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    for tech in techniques:
        print(f"  [{tech}] running...")
        metrics, probs = run_ada_technique(X_train, y_train, X_test, y_test, tech, v)
        metrics['Version']   = v
        metrics['Technique'] = tech_names[tech]
        all_results.append(metrics)
        np.save(os.path.join(RESULTS_DIR, f'probs_ada_{tech}_version_{v}.npy'), probs)
        print_metrics(metrics, label=f"ADA-{tech} V{v}")

print("\n[Exp10-ADA] All complete.")


[Exp10-ADA] Version A
  [baseline] running...


d:\higgs_Research\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[ADA-baseline VA] AUC=0.8006 | F1=0.7411 | Signal_Sig=167.4141 | Train_Time=1732.91s
  [class_weights] running...


d:\higgs_Research\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[ADA-class_weights VA] AUC=0.8006 | F1=0.7322 | Signal_Sig=167.4141 | Train_Time=494.34s
  [smote] running...


...

[ADA-smote VC] AUC=0.6944 | F1=0.0983 | Signal_Sig=4.2956 | Train_Time=520.81s
  [focal_loss] running...
    [NOTE] AdaBoost does not support focal loss — replacing with class_weight='balanced' equivalent via sample_weight.


d:\higgs_Research\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[ADA-focal_loss VC] AUC=0.7097 | F1=0.0913 | Signal_Sig=4.3025 | Train_Time=172.91s
  [threshold] running...


d:\higgs_Research\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[ADA-threshold VC] AUC=0.7023 | F1=0.0878 | Signal_Sig=4.3032 | Train_Time=148.52s

[Exp10-ADA] All complete.


In [4]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment10_adaboost.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.800575 0.741149           167.414142         1732.91
      A  Class Weights 0.800575 0.732181           167.414142          494.34
      A          SMOTE 0.798691 0.733117           167.414142          522.82
      A     Focal Loss 0.800575 0.732181           167.414142          379.63
      A Threshold Opt. 0.800030 0.740180           167.414142          318.31
      B       Baseline 0.787956 0.224830            20.700568          180.65
      B  Class Weights 0.783649 0.318371            20.689162          185.33
      B          SMOTE 0.741759 0.319764            20.683766          509.27
      B     Focal Loss 0.783649 0.318371            20.689162          185.25
      B Threshold Opt. 0.775964 0.212012            20.692162          160.81
      C       Baseline 0.707759 0.036715             4.310034          172.64
      C  Class Weights 0.709719 0.091311             4.302549   